# Joan Tryhard

### Imports

In [2]:
import pandas as pd
import sklearn
import imblearn
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix

### Get Data and Preprocess

In [3]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
import pandas as pd

# Load original data, keeping the 'language' column for filtering
train_orig = pd.read_csv("data/train_dataset_processed.csv")
test_orig = pd.read_csv("data/test_dataset_processed.csv")

# Get unique languages from training data
languages = train_orig['language'].unique()

# --- One-hot encode 'language' for the feature set ---
enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore') # Using sparse_output=False for easier DataFrame creation
# Fit encoder on training data languages
enc.fit(train_orig[['language']])

# Transform train data
language_encoded_train = enc.transform(train_orig[['language']])
language_df_train = pd.DataFrame(language_encoded_train,
                                 columns=enc.get_feature_names_out(['language']),
                                 index=train_orig.index)
train_processed = pd.concat([train_orig.drop(columns=['language']), language_df_train], axis=1)

# Transform test data
language_encoded_test = enc.transform(test_orig[['language']])
language_df_test = pd.DataFrame(language_encoded_test,
                                columns=enc.get_feature_names_out(['language']),
                                index=test_orig.index)
test_processed = pd.concat([test_orig.drop(columns=['language']), language_df_test], axis=1)

# Prepare full data and labels (these will be filtered per language or used for fallback)
X_full = train_processed.drop(columns=['root'])
y_full = train_processed['root']
X_test_processed = test_processed # This is the X_test to make predictions on

print("Data preprocessing complete.")
print(f"X_full shape: {X_full.shape}")
print(f"y_full shape: {y_full.shape}")
print(f"X_test_processed shape: {X_test_processed.shape}")

Data preprocessing complete.
X_full shape: (197479, 35)
y_full shape: (197479,)
X_test_processed shape: (194648, 35)


## Models

### Unimodel Random Forest

In [6]:
from sklearn.ensemble import RandomForestClassifier

trained_models = {}
all_language_prob_predictions = []

# Columns to drop for language-specific models (the OHE language features)
lang_cols_to_drop = [col for col in X_full.columns if col.startswith('language_')]

for lang in tqdm(languages, desc="Training models per language"):
    # 1. Filter data for the current language
    train_lang_indices = train_orig[train_orig['language'] == lang].index
    
    # Prepare features for this language's model (dropping OHE language columns)
    X_train_lang = X_full.loc[train_lang_indices].drop(columns=lang_cols_to_drop, errors='ignore')
    y_train_lang = y_full.loc[train_lang_indices]

    if X_train_lang.empty or len(y_train_lang.unique()) < 2: # Skip if no data or only one class
        print(f"Skipping language {lang} due to insufficient/unsuitable training data ({len(X_train_lang)} samples, {len(y_train_lang.unique())} classes).")
        continue

    # 2. Train a model for the current language
    print(f"Training model for {lang} ({len(X_train_lang)} samples)... ")
    clf_lang = RandomForestClassifier(random_state=42) # Add random_state for reproducibility
    clf_lang.fit(X_train_lang, y_train_lang)
    trained_models[lang] = clf_lang
    print(f"Model for {lang} trained.")

    # 3. Make predictions for this language on the test set
    test_lang_indices = test_orig[test_orig['language'] == lang].index
    if not test_lang_indices.empty:
        X_test_lang = X_test_processed.loc[test_lang_indices].drop(columns=lang_cols_to_drop, errors='ignore')
        
        if not X_test_lang.empty:
            prob_predictions_lang = clf_lang.predict_proba(X_test_lang)
            
            # Ensure prob_predictions_lang_df has both 0 and 1 columns
            prob_dict = {}
            model_classes = list(clf_lang.classes_)
            if 0 in model_classes and 1 in model_classes:
                idx_0 = model_classes.index(0)
                idx_1 = model_classes.index(1)
                prob_dict[0] = prob_predictions_lang[:, idx_0]
                prob_dict[1] = prob_predictions_lang[:, idx_1]
            elif 0 in model_classes: # Only class 0 predicted by model (or seen in y_train_lang)
                prob_dict[0] = prob_predictions_lang[:, model_classes.index(0)] if prob_predictions_lang.ndim > 1 else prob_predictions_lang
                prob_dict[1] = np.zeros_like(prob_dict[0])
            elif 1 in model_classes: # Only class 1 predicted by model
                prob_dict[1] = prob_predictions_lang[:, model_classes.index(1)] if prob_predictions_lang.ndim > 1 else prob_predictions_lang
                prob_dict[0] = np.zeros_like(prob_dict[1])
            else: # Should not happen if model trained and data available
                print(f"Warning: Model for {lang} did not produce expected class probabilities. Classes: {model_classes}. Setting to 0.")
                prob_dict[0] = np.zeros(len(X_test_lang))
                prob_dict[1] = np.zeros(len(X_test_lang))

            prob_predictions_lang_df = pd.DataFrame(prob_dict, index=X_test_lang.index)
            all_language_prob_predictions.append(prob_predictions_lang_df)
    else:
        print(f"No test samples for language {lang}.")

# Combine all predictions
if all_language_prob_predictions:
    prob_predictions_df = pd.concat(all_language_prob_predictions).sort_index()
else:
    print("Warning: No language-specific predictions were made. Initializing empty prob_predictions_df for fallback.")
    # Create a DataFrame with the full index of X_test_processed and default 0.0 probabilities
    # This ensures that if NO language-specific models could make predictions, the fallback has a base to work from.
    prob_predictions_df = pd.DataFrame(0.0, index=X_test_processed.index, columns=[0, 1])


# Fallback for languages in test but not in train (or no model trained or model produced all zeros)
# missing_indices are those in X_test_processed not yet in prob_predictions_df
missing_indices = X_test_processed.index.difference(prob_predictions_df.index)

# Also consider cases where a model might have predicted [0,0] (e.g., if only one class was in its training slice or some edge case)
# These should also be handled by the fallback.
if not prob_predictions_df.empty:
   predicted_indices_for_fallback = prob_predictions_df[prob_predictions_df.sum(axis=1) == 0].index
   missing_indices = missing_indices.union(predicted_indices_for_fallback)

if not missing_indices.empty:
    print(f"Missing or zero predictions for {len(missing_indices)} samples. Applying a fallback model.")
    # Option 1: Train a global fallback model (uses OHE language features)
    print("Training a global fallback model...")
    fallback_clf = RandomForestClassifier(random_state=42)
    fallback_clf.fit(X_full, y_full) # Train on all data, including OHE lang features
    
    X_test_missing = X_test_processed.loc[missing_indices]
    if not X_test_missing.empty:
        prob_predictions_missing = fallback_clf.predict_proba(X_test_missing)
        
        prob_dict_fallback = {}
        fallback_model_classes = list(fallback_clf.classes_)
        if 0 in fallback_model_classes and 1 in fallback_model_classes:
            idx_0_fb = fallback_model_classes.index(0)
            idx_1_fb = fallback_model_classes.index(1)
            prob_dict_fallback[0] = prob_predictions_missing[:, idx_0_fb]
            prob_dict_fallback[1] = prob_predictions_missing[:, idx_1_fb]
        elif 0 in fallback_model_classes:
            prob_dict_fallback[0] = prob_predictions_missing[:, fallback_model_classes.index(0)] if prob_predictions_missing.ndim > 1 else prob_predictions_missing
            prob_dict_fallback[1] = np.zeros_like(prob_dict_fallback[0])
        elif 1 in fallback_model_classes:
            prob_dict_fallback[1] = prob_predictions_missing[:, fallback_model_classes.index(1)] if prob_predictions_missing.ndim > 1 else prob_predictions_missing
            prob_dict_fallback[0] = np.zeros_like(prob_dict_fallback[1])
        else:
            print("Warning: Fallback model did not produce expected class probabilities. Setting to 0.")
            prob_dict_fallback[0] = np.zeros(len(X_test_missing))
            prob_dict_fallback[1] = np.zeros(len(X_test_missing))

        prob_predictions_missing_df = pd.DataFrame(prob_dict_fallback, index=missing_indices)
        
        # Update existing prob_predictions_df.
        # `update` modifies in place for existing indices and doesn't add new rows.
        prob_predictions_df.update(prob_predictions_missing_df)
        
        # For any indices in prob_predictions_missing_df that were not originally in prob_predictions_df
        # (e.g., if prob_predictions_df was empty before this, or had a different set of initial predictions)
        # we need to concatenate them.
        newly_added_indices = prob_predictions_missing_df.index.difference(prob_predictions_df.index)
        if not newly_added_indices.empty:
            prob_predictions_df = pd.concat([prob_predictions_df, prob_predictions_missing_df.loc[newly_added_indices]])
            
        prob_predictions_df = prob_predictions_df.sort_index() # Ensure index is sorted after potential concat
        print("Fallback predictions applied.")

# Final check: Ensure prob_predictions_df has predictions for all test samples.
# If any are still missing (shouldn't happen if logic is correct), fill with 0.5/0.5.
if len(prob_predictions_df) != len(X_test_processed):
    print(f"Warning: Prediction count ({len(prob_predictions_df)}) does not match test set size ({len(X_test_processed)}). Re-aligning.")
    # Reindex to match X_test_processed, filling missing with 0.5 for safety (though this indicates a prior issue)
    prob_predictions_df = prob_predictions_df.reindex(X_test_processed.index)
    if prob_predictions_df.isnull().values.any(): # If reindex introduced NaNs
         prob_predictions_df.fillna(0.5, inplace=True) # Fill any NaNs (e.g., from reindex)
         print("Filled NaNs introduced by re-alignment with 0.5/0.5.")


# Ensure columns are named 0 and 1 (for class 0 and class 1 probabilities)
prob_predictions_df.columns = [0, 1]


print("Final prob_predictions_df shape:", prob_predictions_df.shape)

Training models per language:   0%|          | 0/21 [00:00<?, ?it/s]

Training model for Japanese (12906 samples)... 


Training models per language:   5%|▍         | 1/21 [00:01<00:24,  1.25s/it]

Model for Japanese trained.
Training model for Finnish (6786 samples)... 


Training models per language:  10%|▉         | 2/21 [00:01<00:16,  1.15it/s]

...

Model for Portuguese trained.
Training model for Arabic (9243 samples)... 


Training models per language:  95%|█████████▌| 20/21 [00:18<00:01,  1.10s/it]

Model for Arabic trained.
Training model for Russian (8224 samples)... 


Training models per language: 100%|██████████| 21/21 [00:19<00:00,  1.08it/s]

Model for Russian trained.
Final prob_predictions_df shape: (194648, 2)


In [7]:
print("Combined probability predictions (head):")
display(prob_predictions_df.head())
print("Combined probability predictions (tail):")
display(prob_predictions_df.tail())
print(f"Is there any NaN in predictions? {prob_predictions_df.isnull().values.any()}")
print(f"Number of rows in test: {len(X_test_processed)}, Number of rows in predictions: {len(prob_predictions_df)}")

Combined probability predictions (head):


,0,1
0,0.93,0.07
1,0.97,0.03
2,0.93,0.07
3,0.81,0.19
4,0.98,0.02


Combined probability predictions (tail):


,0,1
194643,1.00,0.00
194644,1.00,0.00
194645,1.00,0.00
194646,0.79,0.21
194647,1.00,0.00


Is there any NaN in predictions? False
Number of rows in test: 194648, Number of rows in predictions: 194648


## Evaluating results

In [8]:
preds = pd.DataFrame({
    'language': test_orig['language'], # Get language from original test data
    'sentence_id': test_orig['sentence_id'],
    'node': test_orig['node'],
    'zero': prob_predictions_df[0],
    'root_prob': prob_predictions_df[1],
}, index=test_orig.index)

print("Predictions DataFrame 'preds' (head):")
display(preds.head())

Predictions DataFrame 'preds' (head):


,language,sentence_id,node,zero,root_prob
0,Japanese,1,5,0.93,0.07
1,Japanese,1,25,0.97,0.03
2,Japanese,1,37,0.93,0.07
3,Japanese,1,2,0.81,0.19
4,Japanese,1,17,0.98,0.02


In [9]:
# Find the index of the row with max 'root_prob' for each group
idx_max_prob_per_group = preds.groupby(['language', 'sentence_id'], sort=False)['root_prob'].idxmax()

# Select these rows from the 'preds' DataFrame
preds_at_max_prob = preds.loc[idx_max_prob_per_group]

# Create the 'preds_grouped' DataFrame by selecting and renaming the 'node' column
preds_grouped = preds_at_max_prob[['node']].copy()
preds_grouped.rename(columns={'node': 'root'}, inplace=True)

# Reset index to be sequential (0, 1, 2, ...)
preds_grouped.reset_index(drop=True, inplace=True)

# Add 'id' column (1-based index for submission)
preds_grouped['id'] = range(1, len(preds_grouped) + 1)

# Ensure columns are in the order ['id', 'root']
preds_grouped = preds_grouped[['id', 'root']]

print("Final grouped predictions for submission (head):")
display(preds_grouped.head())

# Save the predictions to a CSV file
preds_grouped.to_csv('data/predictions_submission_multimodel.csv', index=False)
print("\nPredictions saved to 'data/predictions_submission_multimodel.csv'")

Final grouped predictions for submission (head):


,id,root
0,1,2
1,2,7
2,3,21
3,4,19
4,5,12



Predictions saved to 'data/predictions_submission_multimodel.csv'


In [10]:
current_predictions = pd.read_csv('data/predictions_submission_multimodel.csv')
print("Loaded submission file (head):")
display(current_predictions.head())

Loaded submission file (head):


,id,root
0,1,2
1,2,7
2,3,21
3,4,19
4,5,12


In [11]:
try:
    kaggle_perfect_predictions = pd.read_csv("data/kaggle_perfect_predictions.csv")
    
    if len(kaggle_perfect_predictions) == len(current_predictions):
        y_true_eval = kaggle_perfect_predictions['root']
        y_pred_eval = current_predictions['root']
        # Ensure labels are consistent for classification_report if some node IDs are missing in either set
        all_labels = sorted(list(set(y_true_eval) | set(y_pred_eval)))
        print(classification_report(y_true_eval, y_pred_eval, labels=all_labels, zero_division=0))
    else:
        print("Skipping classification_report: Row count mismatch between perfect predictions and current predictions.")
        print(f"Kaggle perfect: {len(kaggle_perfect_predictions)}, Current: {len(current_predictions)}")
except FileNotFoundError:
    print("Kaggle perfect predictions file not found. Skipping classification report.")
except Exception as e:
    print(f"An error occurred during classification report generation: {e}")

              precision    recall  f1-score   support

           1       0.26      0.31      0.28       690
           2       0.29      0.30      0.29       675
           3       0.27      0.27      0.27       687
           4       0.25      0.27      0.26       641
           5       0.29      0.28      0.28       693
           6       0.29      0.28      0.28       626
           7       0.26      0.25      0.26       653
           8       0.25      0.23      0.24       607
           9       0.26      0.25      0.25       547
          10       0.23      0.22      0.23       510
          11       0.26      0.26      0.26       491
          12       0.24      0.24      0.24       407
          13       0.25      0.23      0.24       390
          14       0.25      0.27      0.26       346
          15       0.23      0.22      0.23       336
          16       0.23      0.22      0.22       312
          17       0.24      0.22      0.23       248
          18       0.21    

In [12]:
def evaluate_model(y_true_df, y_pred_df):
    # Assuming y_true_df and y_pred_df are DataFrames with 'root' and matching length
    if 'root' not in y_true_df.columns or 'root' not in y_pred_df.columns:
        print("Error: 'root' column missing in one of the dataframes.")
        return
    if len(y_true_df) != len(y_pred_df):
        print("Error: Length mismatch between true and predicted values.")
        print(f"  Length of true values: {len(y_true_df)}")
        print(f"  Length of predicted values: {len(y_pred_df)}")
        return
        
    y_true = y_true_df['root']
    y_pred = y_pred_df['root']
    correct_predictions = (y_true == y_pred).sum()
    total_predictions = len(y_true)
    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
    print(f"Number of correct predictions: {correct_predictions} / {total_predictions}")
    print(f"Evaluation accuracy: {accuracy:.4f}")

try:
    # Reload for this function to ensure clean state
    kaggle_perfect_predictions_eval = pd.read_csv("data/kaggle_perfect_predictions.csv")
    current_predictions_eval = pd.read_csv('data/predictions_submission_multimodel.csv')
    evaluate_model(kaggle_perfect_predictions_eval, current_predictions_eval)
except FileNotFoundError:
    print("One of the prediction files not found. Skipping custom evaluation.")
except Exception as e:
    print(f"An error occurred during custom evaluation: {e}")

Number of correct predictions: 2646 / 10395
Evaluation accuracy: 0.2545
